# 🛍️ E-Commerce Analytics: End-to-End Business Case

## 🔹 Problem Context
E-commerce platforms like **Myntra** generate millions of rows of data. Direct interpretation is impossible without structured analysis. This project uses the **Olist Brazilian E-commerce dataset** to:
1. Identify High-Value Customers (RFM).
2. Forecast Sales Revenue.
3. Understand Customer Satisfaction drivers.

---

## 🔹 PART 1: DATA ENGINEERING
We start by merging the relational tables and performing data cleaning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# 1. Load Data
customers = pd.read_csv('data/olist_customers_dataset.csv')
orders = pd.read_csv('data/olist_orders_dataset.csv')
items = pd.read_csv('data/olist_order_items_dataset.csv')
payments = pd.read_csv('data/olist_order_payments_dataset.csv')
products = pd.read_csv('data/olist_products_dataset.csv')
reviews = pd.read_csv('data/olist_order_reviews_dataset.csv')
translation = pd.read_csv('data/product_category_name_translation.csv')

# 2. Merge Tables
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(items, on='order_id', how='left')
df = df.merge(payments, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(reviews, on='order_id', how='left')
df = df.merge(translation, on='product_category_name', how='left')

print(f"Raw dataset merged into {df.shape[0]} rows and {df.shape[1]} columns.")

# 3. Cleaning & Features
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['product_category_name_english'] = df['product_category_name_english'].fillna('other')

df.head()

## 🔹 PART 2: EXPLORATORY DATA ANALYSIS (EDA)
Visualizing revenue trends and identifying the most profitable categories.

In [ ]:
sns.set(style="whitegrid")
plt.figure(figsize=(15, 6))

# Monthly Revenue Trend
monthly_sales = df.groupby(['order_year', 'order_month'])['payment_value'].sum().reset_index()
monthly_sales['date'] = monthly_sales['order_year'].astype(str) + '-' + monthly_sales['order_month'].astype(str)
sns.lineplot(data=monthly_sales, x='date', y='payment_value', marker='o', color='purple')
plt.title('Monthly Sales Revenue Growth')
plt.xticks(rotation=45)
plt.show()

## 🔹 PART 3: CUSTOMER SEGMENTATION (RFM)
We use **Recency, Frequency, and Monetary** (RFM) to cluster our customers.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# RFM Aggregation
max_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (max_date - x.max()).days,
    'order_id': 'count',
    'payment_value': 'sum'
}).rename(columns={'order_purchase_timestamp': 'Recency', 'order_id': 'Frequency', 'payment_value': 'Monetary'})

# Scaling and Clustering
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

print("Cluster Profiles:")
print(rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean())

## 🔹 PART 4: SALES PREDICTION MODELS
Comparing models to predict the value of future orders.

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

# Feature Selection
features = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'review_score']
ml_df = df.dropna(subset=features + ['payment_value'])

X = ml_df[features]
y = ml_df['payment_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training XGBoost
model = XGBRegressor(n_estimators=100)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"XGBoost R2 Score: {r2_score(y_test, preds):.4f}")

## 🔹 PART 5: ETHICAL CONSIDERATIONS
- **Bias**: Data is from Brazil; results may not directly generalize to APAC or US without retraining.
- **Privacy**: Anonymization of customer details is crucial in real-world deployments.